# Price Elasticity & Scenario Testing
**Marketing Analytics Module 3** — Scan\*Pro log-log OLS · Price elasticity · What-if scenario simulation

## The question

Which SKUs are most sensitive to price changes? How would a 10% price increase or decrease impact demand and revenue? This module answers these questions to guide pricing strategy.

## Why we need a model

Simply observing that "weeks with lower prices had higher sales" conflates the price effect with promotions (products are often discounted *and* featured together), seasonality (prices may drop during high-demand periods), and long-run demand trends. We need a regression model to isolate the *pure* price sensitivity after controlling for these confounders.

## Why this specific model

We use the Scan\*Pro log-log OLS specification because it produces a coefficient (β₁) that *directly equals* the price elasticity — no transformation needed. This is a standard microeconomic result:

$$\log(\text{sales}) = \beta_0 + \underbrace{\beta_1}_{\text{= elasticity}} \cdot \log(\text{price}) + \beta_2 \cdot \text{feat\_main\_page} + \beta_3 \cdot \text{trend} + \sum_{m} \gamma_m \cdot \text{month}_{m} + \varepsilon$$

### Key design choice: no lagged prices (unlike the Promotion tab)

The Promotion Effectiveness module includes lagged prices (price_{t-1}, price_{t-2}) to capture reference-price formation and stockpiling effects, because omitting them would bias the promotion coefficient β₂.

This module deliberately **excludes** lagged prices because the goal is different. We want a single **steady-state** elasticity number for what-if scenarios: "what happens if we permanently raise price by 10%?" Including lags would split the price effect into short-run and long-run components, complicating the scenario simulation — which lag structure do you propagate forward? The contemporaneous-only specification gives one clean elasticity that directly feeds into the scenario calculator.

### Why each feature is in the model

| Feature | Role | What happens if we omit it |
|---------|------|---------------------------|
| **log(price)** | The variable of interest — β₁ = elasticity | — |
| **feat_main_page** | Controls for promotions | β₁ is biased if promotions coincide with price cuts (captures promo + price bundled) |
| **trend** | Controls for secular demand growth/decline | β₁ is biased if prices trend downward while demand grows (captures time trend as price effect) |
| **month dummies** | Controls for seasonal demand patterns | β₁ is biased if prices follow seasonal cycles (e.g. lower in December when demand is naturally high) |

### Why log-log and not linear?

In a log-log model: β₁ = ∂ln(Q)/∂ln(P) = (%ΔQ)/(%ΔP) — this is the textbook definition of price elasticity. A linear model would give ∂Q/∂P (units per pound), which varies with the price level and doesn't generalise to percentage scenario simulations. The constant-elasticity assumption is a standard simplification in marketing mix modelling (Hanssens et al., 2001).

### How scenarios are computed

From the estimated elasticity ε:
$$Q_{\text{new}} = Q_0 \times \left(\frac{P_{\text{new}}}{P_0}\right)^{\varepsilon}, \quad R_{\text{new}} = P_{\text{new}} \times Q_{\text{new}}$$

- **Elastic (|ε| > 1):** A 10% price rise causes >10% demand drop → revenue falls → consider lowering price
- **Inelastic (|ε| < 1):** A 10% price rise causes <10% demand drop → revenue rises → consider raising price

**Reference:** Van Heerde, H.J., Leeflang, P.S.H., & Wittink, D.R. (2002). *Schmalenbach Business Review*, 54, 198–220.

In [1]:
# Load all libraries needed for modelling, charts, and interactive widgets
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

pio.templates.default = "plotly_white"

# ── Single blue accent palette (professional/corporate) ───────────────────────
# Inspired by Tufte's data-ink ratio principle — colour only where it carries meaning
BLUE     = "#2563EB"          # Primary accent — key data points, best model
BLUE_LT  = "rgba(37,99,235,0.10)"  # CI band fill — extremely subtle
SLATE    = "#0f172a"          # Near-black — actual/observed data
GREY_LG  = "#cbd5e1"          # Light grey — historical / secondary traces
GREY_MD  = "#94a3b8"          # Mid grey   — gridlines, annotations
RED      = "#DC2626"          # Negative Δ revenue only
GREEN    = "#16a34a"          # Positive Δ revenue only

# ── KPI card helpers — identical style to Module 1 & 2 ───────────────────────
def metric_card(label, value, sub=""):
    return f'''<div style="flex:1;min-width:160px;background:#f8fafc;
    border:1px solid #e2e8f0;border-radius:12px;padding:1.1rem 1.4rem;">
    <div style="font-size:0.72rem;color:#64748b;text-transform:uppercase;
    letter-spacing:0.05em;font-weight:500;">{label}</div>
    <div style="font-size:1.45rem;font-weight:700;color:#0f172a;
    margin-top:0.2rem;font-family:monospace;">{value}</div>
    <div style="font-size:0.78rem;color:#64748b;margin-top:0.15rem;">{sub}</div>
    </div>'''

def metric_row(cards):
    inner = "".join(cards)
    return HTML(f'<div style="display:flex;gap:1rem;margin:1rem 0;'
                f'flex-wrap:wrap;">{inner}</div>')




In [2]:
# Load datasets and build SKU name lookup from raw data
proc = pd.read_csv("../data/data_processed.csv")
proc["week"] = pd.to_datetime(proc["week"])
proc = proc.sort_values(["sku", "week"]).reset_index(drop=True)

# Build SKU metadata from raw data (names, colours, categories)
raw = pd.read_csv("../data/data_raw.csv")
raw["week"] = pd.to_datetime(raw["week"])
raw["feat_main_page"] = raw["feat_main_page"].astype(str).str.lower().eq("true").astype(int)

sku_meta = raw.groupby("sku").agg(
    functionality=("functionality", "first"),
    color=("color", "first"),
    vendor=("vendor", "first"),
    avg_price=("price", "mean"),
    avg_sales=("weekly_sales", "mean"),
    promo_rate=("feat_main_page", "mean"),
).reset_index()

# Clean category — strip numeric prefix (e.g. "06.Mobile phone accessories" → "Mobile phone accessories")
sku_meta["category"] = sku_meta["functionality"].str.replace(
    r"^\d+\.", "", regex=True).str.strip()

# Human-readable display name: "Mobile phone accessories (Black)"
sku_meta["display_name"] = sku_meta.apply(
    lambda r: f"{r['category']} ({r['color'].title()})", axis=1)

# Full dropdown label: "SKU 1 — Mobile phone accessories (Black)"
sku_meta["label"] = sku_meta.apply(
    lambda r: f"SKU {r['sku']} — {r['display_name']}", axis=1)

def sku_label(sku_id):
    row = sku_meta[sku_meta["sku"] == sku_id]
    return row["label"].iloc[0] if len(row) > 0 else f"SKU {sku_id}"

def sku_category(sku_id):
    row = sku_meta[sku_meta["sku"] == sku_id]
    return row["category"].iloc[0] if len(row) > 0 else "Unknown"

MONTH_COLS = [f"month_{i}" for i in range(2, 13)]
SKUS       = sorted(proc["sku"].unique())
SCENARIOS  = [-0.30, -0.20, -0.10, -0.05, 0.05, 0.10, 0.20, 0.30]

print(f"Loaded {proc['sku'].nunique()} SKUs · {proc['week'].nunique()} weeks")
print(f"   Date range: {proc['week'].min().date()} → {proc['week'].max().date()}")
print(f"\nSample SKU names:")
for s in [1, 8, 10, 25, 43]:
    print(f"   {sku_label(s)}")


Loaded 44 SKUs · 98 weeks
   Date range: 2016-11-14 → 2018-09-24

Sample SKU names:
   SKU 1 — Mobile phone accessories (Black)
   SKU 8 — Bluetooth speakers (Black)
   SKU 10 — VR headset (White)
   SKU 25 — Selfie sticks (Green)
   SKU 43 — Fitness trackers (Gold)


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# SCAN*PRO MODEL FOR PRICE ELASTICITY
# ══════════════════════════════════════════════════════════════════════════════
# Goal: estimate β₁ (price elasticity) per SKU, controlling for confounders.
# This is a coefficient estimation problem, not prediction.
# Model: log(sales) = β₀ + β₁·log(price) + β₂·feat + β₃·trend + Σγ·month + ε
#
# Key difference from Promotion tab: NO lagged prices here.
# Promotion tab needs lags to debias β₂ (Van Heerde et al., 2002).
# This tab wants steady-state elasticity for what-if scenarios —
# a single β₁ that feeds directly into Q_new = Q₀ × (P_new/P₀)^ε.
# Including lags would fragment elasticity into short/long-run components,
# complicating scenario simulation.

def fit_scanpro(sku_id):
    """Fit Scan*Pro for one SKU. Returns result dict or None."""
    df = proc[proc["sku"] == sku_id].sort_values("week").copy()
    df = df[df["weekly_sales"] > 0]
    if len(df) < 15 or df["feat_main_page"].nunique() < 2:
        return None

    y = np.log(df["weekly_sales"].values)
    X = pd.DataFrame(index=df.index)
    X["log_price"]      = np.log(df["price"].clip(lower=0.01))
    X["feat_main_page"] = df["feat_main_page"]
    X["trend"]          = df["trend"]
    for m in MONTH_COLS:
        if m in df.columns:
            X[m] = df[m]
    X = sm.add_constant(X, has_constant="add")

    try:
        # HC1 robust standard errors account for heteroscedasticity
        # (variance that changes with price level — common in sales data)
        model = sm.OLS(y, X).fit(cov_type='HC1')
    except Exception:
        return None

    elast      = float(model.params.get("log_price", np.nan))
    elast_pval = float(model.pvalues.get("log_price", np.nan))
    if np.isnan(elast):
        return None

    P0 = float(df["price"].mean())
    Q0 = float(df["weekly_sales"].mean())
    R0 = P0 * Q0

    # Continuous price curve for demand/revenue charts
    price_range   = np.linspace(P0 * 0.20, P0 * 3.0, 400)
    demand_curve  = np.clip(Q0 * (price_range / P0) ** elast, 0, None)
    revenue_curve = price_range * demand_curve
    opt_idx       = int(np.argmax(revenue_curve))

    # What-if scenario table
    rows = []
    for pct in SCENARIOS:
        P1 = P0 * (1 + pct)
        Q1 = max(Q0 * ((P1 / P0) ** elast), 0)
        R1 = P1 * Q1
        rows.append({
            "Scenario"        : f"{'↑' if pct>0 else '↓'} {abs(int(pct*100))}%",
            "pct"             : pct,
            "New Price (£)"   : round(P1, 2),
            "Δ Demand (units)": round(Q1 - Q0, 1),
            "Δ Demand (%)"    : round((Q1-Q0)/Q0*100, 1) if Q0 > 0 else 0,
            "New Revenue (£)" : round(R1, 2),
            "Δ Revenue (£)"   : round(R1-R0, 2),
            "Δ Revenue (%)"   : round((R1-R0)/R0*100, 1) if R0 > 0 else 0,
        })

    return {
        "sku"            : int(sku_id),
        "name"           : sku_label(int(sku_id)),
        "category"       : sku_category(int(sku_id)),
        "elasticity"     : round(elast, 4),
        "elast_pval"     : round(elast_pval, 4),
        "elast_sig"      : bool(elast_pval < 0.05),
        "r2"             : round(float(model.rsquared), 4),
        "avg_price"      : round(P0, 2),
        "avg_sales"      : round(Q0, 1),
        "avg_revenue"    : round(R0, 2),
        "price_range"    : price_range,
        "demand_curve"   : demand_curve,
        "revenue_curve"  : revenue_curve,
        "optimal_price"  : round(float(price_range[opt_idx]), 2),
        "optimal_revenue": round(float(revenue_curve[opt_idx]), 2),
        "scenario_df"    : pd.DataFrame(rows),
        "weeks"          : df["week"].values,
        "actual"         : df["weekly_sales"].values,
    }

# Run on all SKUs
print(" Fitting Scan*Pro for all 44 SKUs...")
results = {}
for sku_id in SKUS:
    r = fit_scanpro(sku_id)
    if r:
        results[sku_id] = r

# Summary DataFrame
results_df = pd.DataFrame([{
    "sku"          : r["sku"],
    "name"         : r["name"],
    "category"     : r["category"],
    "elasticity"   : r["elasticity"],
    "elast_pval"   : r["elast_pval"],
    "elast_sig"    : r["elast_sig"],
    "r2"           : r["r2"],
    "avg_price"    : r["avg_price"],
    "avg_sales"    : r["avg_sales"],
    "avg_revenue"  : r["avg_revenue"],
    "optimal_price": r["optimal_price"],
} for r in results.values()]).sort_values("elasticity").reset_index(drop=True)

print(f" Fitted {len(results_df)} SKUs")


 Fitting Scan*Pro for all 44 SKUs...
 Fitted 44 SKUs


In [4]:
# ── Elasticity classification ────────────────────────────────────────────────
# Categorise SKUs by price sensitivity for actionable strategy recommendations

def classify_elasticity(e):
    if abs(e) < 1:
        return 'Inelastic'
    elif abs(e) < 2:
        return 'Moderately Elastic'
    else:
        return 'Highly Elastic'

results_df['elast_category'] = results_df['elasticity'].apply(classify_elasticity)

print('Elasticity distribution:')
for cat in ['Highly Elastic', 'Moderately Elastic', 'Inelastic']:
    n = (results_df['elast_category'] == cat).sum()
    print(f'  {cat}: {n} SKUs ({n/len(results_df)*100:.0f}%)')

Elasticity distribution:
  Highly Elastic: 16 SKUs (36%)
  Moderately Elastic: 16 SKUs (36%)
  Inelastic: 12 SKUs (27%)


In [5]:
# ── Portfolio overview KPIs ───────────────────────────────────────────────────
n_elastic   = int((results_df["elasticity"] < -1).sum())
n_inelastic = int(((results_df["elasticity"] >= -1) & (results_df["elasticity"] < 0)).sum())
n_sig       = int(results_df["elast_sig"].sum())
avg_elast   = round(results_df["elasticity"].mean(), 3)
most_sens   = results_df.loc[results_df["elasticity"].idxmin()]
least_sens  = results_df.loc[results_df["elasticity"].idxmax()]

display(metric_row([
    metric_card("SKUs modelled",        f"{len(results_df)}",
                "with sufficient data"),
    metric_card("Elastic  |ε| > 1",    f"{n_elastic} / {len(results_df)}",
                "price-sensitive demand"),
    metric_card("Inelastic  |ε| < 1",  f"{n_inelastic} / {len(results_df)}",
                "price-insensitive demand"),
    metric_card("Significant  p<0.05", f"{n_sig} / {len(results_df)}",
                "statistically reliable ε"),
    metric_card("Avg elasticity",       f"{avg_elast}",
                "across all SKUs"),
    metric_card("Avg model R²",         f"{results_df['r2'].mean():.3f}",
                "Scan*Pro goodness of fit"),
]))

display(metric_row([
    metric_card("Most price-sensitive",
                f"SKU {int(most_sens['sku'])}",
                f"{most_sens['name'].split('—')[-1].strip()}  |  ε = {most_sens['elasticity']:.3f}"),
    metric_card("Least price-sensitive",
                f"SKU {int(least_sens['sku'])}",
                f"{least_sens['name'].split('—')[-1].strip()}  |  ε = {least_sens['elasticity']:.3f}"),
]))


## Portfolio overview

In [6]:
# ── Elasticity ranking — all SKUs ────────────────────────────────────────────
# Horizontal bar chart ordered by elasticity.
# Blue = significant (p<0.05) · Grey = not significant · dashed line = unit elastic

df_plot = results_df.copy()
df_plot["bar_color"] = df_plot.apply(
    lambda r: BLUE if r["elast_sig"] else GREY_MD, axis=1)
df_plot["label"] = df_plot.apply(
    lambda r: f"SKU {int(r['sku'])} — {r['name'].split('—')[-1].strip()}"
              + (" ★" if r["elast_sig"] else ""), axis=1)

fig = go.Figure(go.Bar(
    y=df_plot["label"],
    x=df_plot["elasticity"],
    orientation="h",
    marker_color=df_plot["bar_color"],
    text=df_plot["elasticity"].apply(lambda v: f"{v:.2f}"),
    textposition="outside",
    textfont=dict(size=9, color=GREY_MD),
    customdata=df_plot[["elast_pval", "avg_price", "avg_sales", "r2",
                         "elast_sig"]].values,
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Elasticity: <b>%{x:.4f}</b><br>"
        "p-value: %{customdata[0]:.4f}"
        "  ·  Significant: %{customdata[4]}<br>"
        "Avg price: £%{customdata[1]:.2f}"
        "  ·  Avg sales: %{customdata[2]:.0f} units/wk<br>"
        "R²: %{customdata[3]:.3f}<extra></extra>"
    ),
))

fig.add_vline(x=-1, line_dash="dash", line_color=GREY_MD, line_width=1.2,
              annotation_text="Unit elastic  ε = −1",
              annotation_font=dict(color=GREY_MD, size=10),
              annotation_position="top right")
fig.add_vline(x=0,  line_dash="dot",  line_color=GREY_MD, line_width=0.8)

fig.update_layout(
    title=dict(text="Price Elasticity per SKU  (Scan*Pro log-log OLS)",
               font=dict(size=14, color=SLATE)),
    height=950,
    margin=dict(l=10, r=80, t=50, b=40),
    xaxis=dict(title="Price Elasticity (ε)", zeroline=True,
               zerolinecolor=GREY_LG, zerolinewidth=1,
               tickfont=dict(color=GREY_MD)),
    yaxis=dict(tickfont=dict(size=10, color=SLATE)),
    showlegend=False,
    plot_bgcolor="white",
)

# Manual legend annotation
fig.add_annotation(
    x=0.99, y=0.01, xref="paper", yref="paper",
    text="■ Blue = significant (p<0.05)  ·  ■ Grey = not significant  ·  ★ = significant",
    showarrow=False, font=dict(size=9, color=GREY_MD),
    bgcolor="white", align="right",
)

fig.show()
print(f"\n{n_elastic} elastic SKUs (|ε|>1)  ·  "
      f"{n_inelastic} inelastic  ·  "
      f"{n_sig} statistically significant")



32 elastic SKUs (|ε|>1)  ·  12 inelastic  ·  39 statistically significant


In [7]:
# ── Revenue impact heatmap — all SKUs × all 8 scenarios ──────────────────────
# Green = revenue gain · Red = revenue loss · white = near-zero change

scenario_labels = [f"{'↑' if p>0 else '↓'}{abs(int(p*100))}%" for p in SCENARIOS]
rows_sorted = results_df.sort_values("elasticity").itertuples()

sku_labels, z_matrix, text_matrix = [], [], []

for row in results_df.sort_values("elasticity").itertuples():
    r     = results[row.sku]
    P0, Q0, R0 = r["avg_price"], r["avg_sales"], r["avg_revenue"]
    zrow, trow = [], []
    for pct in SCENARIOS:
        P1   = P0 * (1 + pct)
        Q1   = max(Q0 * ((P1 / P0) ** r["elasticity"]), 0)
        dpct = (P1*Q1 - R0) / R0 * 100 if R0 > 0 else 0
        zrow.append(round(dpct, 1))
        trow.append(f"{dpct:+.1f}%")
    sku_labels.append(r["name"].replace("SKU ", ""))  # shorter label
    z_matrix.append(zrow)
    text_matrix.append(trow)

fig = go.Figure(go.Heatmap(
    z=z_matrix,
    x=scenario_labels,
    y=sku_labels,
    text=text_matrix,
    texttemplate="%{text}",
    textfont=dict(size=8, color="#334155"),
    colorscale=[
        [0.00, "#DC2626"],
        [0.40, "#FCA5A5"],
        [0.50, "#F8FAFC"],
        [0.60, "#86EFAC"],
        [1.00, "#16a34a"],
    ],
    zmid=0,
    colorbar=dict(title="ΔRevenue (%)", ticksuffix="%", len=0.7,
                  tickfont=dict(size=10)),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Scenario: %{x}<br>"
        "Revenue change: <b>%{z:+.1f}%</b><extra></extra>"
    ),
))

fig.update_layout(
    title=dict(text="Revenue Impact Heatmap  —  All SKUs × All Price Scenarios",
               font=dict(size=14, color=SLATE)),
    height=1000,
    margin=dict(l=10, r=110, t=50, b=60),
    xaxis=dict(title="Price Change Scenario", side="top",
               tickfont=dict(size=11)),
    yaxis=dict(tickfont=dict(size=9), autorange="reversed"),
    paper_bgcolor="white",
)

fig.show()
print("Tip: SKUs at the top are most elastic — they react strongly to price changes.")
print("     SKUs showing green at '+10%' are inelastic — they can absorb price rises.")


Tip: SKUs at the top are most elastic — they react strongly to price changes.
     SKUs showing green at '+10%' are inelastic — they can absorb price rises.


## All-SKU price strategy summary

In [8]:
# ── All-SKU strategy table ───────────────────────────────────────────────────
# Shows ±10% revenue impact and recommended strategy per SKU

summary_rows = []
for _, row in results_df.iterrows():
    r  = results[row["sku"]]
    ε  = r["elasticity"]
    P0, Q0, R0 = r["avg_price"], r["avg_sales"], r["avg_revenue"]
    Pup, Pdn   = P0*1.10, P0*0.90
    Rup = Pup * max(Q0*((Pup/P0)**ε), 0)
    Rdn = Pdn * max(Q0*((Pdn/P0)**ε), 0)
    dUp = round((Rup-R0)/R0*100, 1) if R0>0 else 0
    dDn = round((Rdn-R0)/R0*100, 1) if R0>0 else 0
    strategy = ("Consider lower price"  if ε < -1
                else "Consider higher price" if ε < 0
                else "Review data / model")
    summary_rows.append({
        "SKU"              : f"SKU {int(row['sku'])}",
        "Product"          : r["name"].split("—")[-1].strip(),
        "Category"         : r["category"],
        "Elasticity (ε)"   : row["elasticity"],
        "Sig."             : "★" if row["elast_sig"] else "",
        "Avg Price (£)"    : row["avg_price"],
        "ΔRev +10% (%)"    : dUp,
        "ΔRev −10% (%)"    : dDn,
        "Optimal Price (£)": row["optimal_price"],
        "Strategy"         : strategy,
    })

summary_df = pd.DataFrame(summary_rows)

def _c(val):
    if isinstance(val, float):
        return f"color:{RED};font-weight:600" if val < 0                else f"color:{GREEN};font-weight:600"
    return ""

styled = (
    summary_df.style
    .applymap(_c, subset=["ΔRev +10% (%)", "ΔRev −10% (%)"])
    .applymap(lambda v: f"color:{BLUE};font-weight:700"
              if isinstance(v, float) and v < -1 else
              (f"color:{GREY_MD}" if isinstance(v, float) else ""),
              subset=["Elasticity (ε)"])
    .format({
        "Elasticity (ε)"   : "{:.4f}",
        "Avg Price (£)"    : "£{:.2f}",
        "ΔRev +10% (%)"    : "{:+.1f}%",
        "ΔRev −10% (%)"    : "{:+.1f}%",
        "Optimal Price (£)": "£{:.2f}",
    })
    .set_properties(**{
        "text-align" : "center",
        "font-size"  : "0.84rem",
        "padding"    : "6px 10px",
    })
    .set_table_styles([{
        "selector": "th",
        "props": [
            ("background-color", "#0f172a"),
            ("color", "white"),
            ("font-size", "0.75rem"),
            ("padding", "8px 10px"),
            ("text-align", "center"),
            ("text-transform", "uppercase"),
            ("letter-spacing", "0.04em"),
        ],
    }])
)

display(styled)
print(f"\n{(summary_df['ΔRev +10% (%)']<0).sum()} SKUs lose revenue at +10% price (elastic)")
print(f"{(summary_df['ΔRev +10% (%)']>0).sum()} SKUs gain revenue at +10% price (inelastic)")


,SKU,Product,Category,Elasticity (ε),Sig.,Avg Price (£),ΔRev +10% (%),ΔRev −10% (%),Optimal Price (£),Strategy
0,SKU 9,Fitness trackers (Black),Fitness trackers,-4.0713,★,£162.19,-25.4%,+38.2%,£32.44,Consider lower price
1,SKU 19,Streaming sticks (Black),Streaming sticks,-4.0068,★,£93.67,-24.9%,+37.2%,£18.73,Consider lower price
2,SKU 33,Selfie sticks (Green),Selfie sticks,-3.5109,★,£9.17,-21.3%,+30.3%,£1.83,Consider lower price
3,SKU 30,Mobile phone accessories (Grey),Mobile phone accessories,-2.9269,★,£19.02,-16.8%,+22.5%,£3.80,Consider lower price
4,SKU 11,Streaming sticks (Black),Streaming sticks,-2.9238,★,£28.72,-16.8%,+22.3%,£5.74,Consider lower price
5,SKU 25,Selfie sticks (Green),Selfie sticks,-2.8389,★,£8.42,-16.1%,+21.3%,£1.68,Consider lower price
6,SKU 34,Portable smartphone chargers (Grey),Portable smartphone chargers,-2.5384,★,£19.83,-13.6%,+17.6%,£3.97,Consider lower price
7,SKU 15,Mobile phone accessories (Red),Mobile phone accessories,-2.5265,★,£17.32,-13.5%,+17.5%,£3.46,Consider lower price
8,SKU 7,Selfie sticks (Blue),Selfie sticks,-2.4623,★,£11.32,-13.0%,+16.6%,£2.26,Consider lower price
9,SKU 24,Flash drives (None),Flash drives,-2.3642,★,£21.26,-12.3%,+15.4%,£4.25,Consider lower price



35 SKUs lose revenue at +10% price (elastic)
8 SKUs gain revenue at +10% price (inelastic)


## Model diagnostics

Before trusting the elasticity estimates, we validate the OLS assumptions:
1. **Multicollinearity (VIF):** Are the regressors too correlated with each other?
2. **Residual patterns:** Are residuals randomly scattered (homoscedastic, normally distributed)?

We show diagnostics for a representative SKU. HC1 robust standard errors already protect against heteroscedasticity, but visual inspection confirms the model is well-specified.

In [9]:
# ── VIF check + residual diagnostics for a representative SKU ────────────────
# Pick the most elastic significant SKU as the representative example
rep_sku = results_df.loc[results_df[results_df['elast_sig']]['elasticity'].idxmin(), 'sku']
rep_sku = int(rep_sku)

display(HTML(f'<h3>Diagnostics for SKU {rep_sku} — '
             f'{sku_label(rep_sku)}</h3>'))

# ── Build the FULL regressor matrix (must match what the model actually uses) ──
df_vif = proc[proc['sku'] == rep_sku].sort_values('week').copy()
df_vif = df_vif[df_vif['weekly_sales'] > 0]

X_full = pd.DataFrame(index=df_vif.index)
X_full['log_price'] = np.log(df_vif['price'].clip(lower=0.01))
X_full['feat_main_page'] = df_vif['feat_main_page']
X_full['trend'] = df_vif['trend']
for m in MONTH_COLS:
    if m in df_vif.columns:
        X_full[m] = df_vif[m].values
X_full = sm.add_constant(X_full)

# ── VIF on the full regressor matrix ──
vif_data = pd.DataFrame({
    'Variable': X_full.columns,
    'VIF': [variance_inflation_factor(X_full.values, i) for i in range(X_full.shape[1])]
})
vif_display = vif_data[vif_data['Variable'] != 'const'].sort_values('VIF', ascending=False)

display(HTML('<h4>Variance Inflation Factors (full model)</h4>'))
display(HTML('<p style="color:#64748b;font-size:.85rem;">'
             'VIF is computed on all regressors in the model — log_price, feat_main_page, '
             'trend, and all 11 month dummies — because collinearity between any pair '
             'affects coefficient estimates.</p>'))
display(vif_display.style.format({'VIF': '{:.1f}'}).background_gradient(subset=['VIF'], cmap='YlOrRd', vmin=1, vmax=10))

max_vif = vif_display['VIF'].max()
max_vif_var = vif_display.iloc[0]['Variable']

if max_vif < 5:
    verdict = '✓ No concerning multicollinearity — all VIF < 5.'
elif max_vif < 10:
    verdict = (f'⚠ Moderate collinearity: {max_vif_var} has VIF = {max_vif:.1f}. '
               'This is acceptable for an estimation model — it widens standard errors '
               'but does not bias the elasticity point estimate. HC1 robust SE compensates.')
else:
    verdict = (f'⚠ High collinearity: {max_vif_var} has VIF = {max_vif:.1f}. '
               'Point estimates remain unbiased but standard errors are inflated. '
               'Interpret significance cautiously for this variable.')

display(HTML(f'<p style="color:#64748b;"><b>Verdict:</b> {verdict}</p>'))

# ── Residual diagnostics (on the full model) ──
y_diag = np.log(df_vif['weekly_sales'].values)
model_diag = sm.OLS(y_diag, X_full).fit(cov_type='HC1')
residuals = model_diag.resid
fitted = model_diag.fittedvalues

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    'Residuals vs Fitted', 'Residual Distribution', 'Q-Q Plot'])

# 1. Residuals vs Fitted
fig.add_trace(go.Scatter(x=fitted, y=residuals, mode='markers',
    marker=dict(color=BLUE, opacity=0.6, size=6),
    hovertemplate='Fitted: %{x:.2f}<br>Residual: %{y:.3f}<extra></extra>'),
    row=1, col=1)
fig.add_hline(y=0, line_color=RED, line_dash='dash', row=1, col=1)

# 2. Histogram of residuals
fig.add_trace(go.Histogram(x=residuals, nbinsx=15,
    marker_color=BLUE, opacity=0.8), row=1, col=2)

# 3. Q-Q plot
qq = stats.probplot(residuals, dist='norm')
fig.add_trace(go.Scatter(x=qq[0][0], y=qq[0][1], mode='markers',
    marker=dict(color=BLUE, size=6, opacity=0.6)), row=1, col=3)
xmin, xmax = qq[0][0].min(), qq[0][0].max()
fig.add_trace(go.Scatter(x=[xmin, xmax],
    y=[qq[1][1] + qq[1][0]*xmin, qq[1][1] + qq[1][0]*xmax],
    mode='lines', line=dict(color=RED, dash='dash')), row=1, col=3)

fig.update_layout(height=350, showlegend=False,
    title=f'Residual Diagnostics — SKU {rep_sku} (HC1 robust SE)',
    margin=dict(t=60, b=40))
fig.update_xaxes(title_text='Fitted values', row=1, col=1)
fig.update_yaxes(title_text='Residuals', row=1, col=1)
fig.update_xaxes(title_text='Residual', row=1, col=2)
fig.update_xaxes(title_text='Theoretical quantiles', row=1, col=3)
fig.update_yaxes(title_text='Sample quantiles', row=1, col=3)
fig.show()

sw = stats.shapiro(residuals)
sw_verdict = '✓ Residuals approximately normal' if sw.pvalue > 0.05 else 'Mild departure from normality (HC1 robust SE compensates)'
display(HTML(f'<p style="color:#64748b;">Shapiro-Wilk normality test: '
             f'W={sw.statistic:.3f}, p={sw.pvalue:.4f} — {sw_verdict}</p>'))


,Variable,VIF
3,trend,7.2
1,log_price,4.8
14,month_12,3.5
13,month_11,3.3
9,month_7,2.5
6,month_4,2.1
10,month_8,2.1
7,month_5,2.0
8,month_6,1.9
12,month_10,1.9


## Interactive SKU deep-dive

Select any SKU to see its demand curve, revenue curve, optimal price, and what-if scenario table.


In [10]:
# ── Interactive SKU drill-down ────────────────────────────────────────────────

sku_dd = widgets.Dropdown(
    options=[(r["name"], sku_id) for sku_id, r in
             sorted(results.items(), key=lambda x: x[0])],
    value=list(results.keys())[0],
    layout=widgets.Layout(width="420px"),
)
out = widgets.Output()

def _render(change=None):
    out.clear_output(wait=True)
    sku_id = sku_dd.value
    r      = results[sku_id]
    ε      = r["elasticity"]
    P0, Q0, R0 = r["avg_price"], r["avg_sales"], r["avg_revenue"]

    with out:
        # ── SKU header ────────────────────────────────────────────────────
        display(HTML(
            f'<div style="margin:.5rem 0 .2rem;">'
            f'<div style="font-size:1.05rem;font-weight:700;color:{SLATE};">'
            f'{r["name"]}</div>'
            f'<div style="font-size:.82rem;color:{GREY_MD};margin-top:.2rem;">'
            f'Category: {r["category"]} &nbsp;·&nbsp; '
            f'Avg price: £{P0:.2f} &nbsp;·&nbsp; '
            f'Avg sales: {Q0:.0f} units/wk &nbsp;·&nbsp; '
            f'Avg revenue: £{R0:,.0f}/wk'
            f'</div></div>'
        ))

        # ── Elasticity interpretation ─────────────────────────────────────
        if ε < -1:
            interp = (f"Elastic  |ε| = {abs(ε):.2f} > 1 — "
                      f"Demand is price-sensitive. "
                      f"A 10% price rise → {abs(ε)*10:.0f}% demand fall. "
                      f"Lowering price is likely to grow revenue.")
        elif ε < 0:
            interp = (f"Inelastic  |ε| = {abs(ε):.2f} < 1 — "
                      f"Demand is price-insensitive. "
                      f"A 10% price rise → {abs(ε)*10:.0f}% demand fall. "
                      f"Raising price is likely to grow revenue.")
        else:
            interp = (f"Positive elasticity  ε = {ε:.2f} — "
                      f"Unusual result. "
                      f"Check model significance (p = {r['elast_pval']:.4f}).")

        display(HTML(
            f'<div style="background:#f8fafc;border-left:3px solid {BLUE};'
            f'padding:.6rem 1rem;margin:.4rem 0 .8rem;border-radius:0 6px 6px 0;'
            f'font-size:.84rem;color:#334155;">{interp}</div>'
        ))

        # ── KPI cards ─────────────────────────────────────────────────────
        sig_str = "★ significant" if r["elast_sig"] else "not significant"
        display(metric_row([
            metric_card("Price elasticity", f"{ε:.4f}",
                        f"p = {r['elast_pval']:.4f}  ·  {sig_str}"),
            metric_card("Model R²",         f"{r['r2']:.4f}",
                        "Scan*Pro goodness of fit"),
            metric_card("Avg price",        f"£{P0:.2f}",
                        f"Avg sales: {Q0:.0f} units/wk"),
            metric_card("Avg revenue",      f"£{R0:,.0f}",
                        "per week at current price"),
            metric_card("Revenue-max price",f"£{r['optimal_price']:.2f}",
                        f"Peak £{r['optimal_revenue']:,.0f}/wk"),
        ]))

        # ── Demand + Revenue curves ───────────────────────────────────────
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=[
                f"Demand Curve  (ε = {ε:.3f})",
                f"Revenue Curve  —  Optimal: £{r['optimal_price']:.2f}",
            ],
            horizontal_spacing=0.10,
        )

        # Demand
        fig.add_trace(go.Scatter(
            x=r["price_range"], y=r["demand_curve"],
            mode="lines", line=dict(color=BLUE, width=2),
            name="Demand", showlegend=False,
            hovertemplate="Price: £%{x:.2f}<br>Demand: %{y:.0f} units<extra></extra>",
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=[P0], y=[Q0], mode="markers",
            marker=dict(size=11, color=SLATE,
                        line=dict(color="white", width=2)),
            name="Current price", showlegend=False,
            hovertemplate=f"Current: £{P0:.2f} → {Q0:.0f} units<extra></extra>",
        ), row=1, col=1)

        # Revenue
        fig.add_trace(go.Scatter(
            x=r["price_range"], y=r["revenue_curve"],
            mode="lines", line=dict(color=GREY_LG, width=2),
            name="Revenue curve", showlegend=False,
            hovertemplate="Price: £%{x:.2f}<br>Revenue: £%{y:,.0f}<extra></extra>",
        ), row=1, col=2)
        fig.add_trace(go.Scatter(
            x=[P0], y=[R0], mode="markers",
            marker=dict(size=11, color=SLATE,
                        line=dict(color="white", width=2)),
            name="Current", showlegend=False,
            hovertemplate=f"Current: £{P0:.2f} → £{R0:,.0f}<extra></extra>",
        ), row=1, col=2)
        fig.add_trace(go.Scatter(
            x=[r["optimal_price"]], y=[r["optimal_revenue"]],
            mode="markers",
            marker=dict(size=14, color=BLUE,
                        symbol="star", line=dict(color="white", width=1.5)),
            name=f"Optimal £{r['optimal_price']:.2f}", showlegend=False,
            hovertemplate=(f"Optimal: £{r['optimal_price']:.2f}"
                           f" → £{r['optimal_revenue']:,.0f}/wk<extra></extra>"),
        ), row=1, col=2)

        for col, ytitle in [(1, "Weekly demand (units)"),
                            (2, "Weekly revenue (£)")]:
            fig.update_xaxes(title_text="Price (£)", row=1, col=col,
                             tickfont=dict(color=GREY_MD))
            fig.update_yaxes(title_text=ytitle, row=1, col=col,
                             tickfont=dict(color=GREY_MD))

        fig.update_layout(
            height=360,
            margin=dict(t=50, b=40, l=60, r=40),
            plot_bgcolor="white",
            paper_bgcolor="white",
        )
        fig.show()

        # ── Scenario waterfall ────────────────────────────────────────────
        sc = r["scenario_df"]
        bar_colors = [RED if v < 0 else GREEN for v in sc["Δ Revenue (£)"]]

        fig2 = go.Figure(go.Bar(
            x=sc["Scenario"],
            y=sc["Δ Revenue (£)"],
            marker_color=bar_colors,
            marker_line_width=0,
            text=[f"£{v:+,.0f}" for v in sc["Δ Revenue (£)"]],
            textposition="outside",
            textfont=dict(size=10, color=GREY_MD),
            customdata=sc[["New Price (£)", "Δ Demand (%)",
                            "New Revenue (£)", "Δ Revenue (%)"]].values,
            hovertemplate=(
                "<b>%{x}</b><br>"
                "New price: £%{customdata[0]:.2f}<br>"
                "Δ Demand: %{customdata[1]:+.1f}%<br>"
                "New revenue: £%{customdata[2]:,.0f}<br>"
                "Δ Revenue: %{customdata[3]:+.1f}%<extra></extra>"
            ),
        ))
        fig2.add_hline(y=0, line_color=GREY_LG, line_width=1)
        fig2.update_layout(
            title=dict(
                text=f"Revenue Impact by Price Scenario  (ε = {ε:.3f})",
                font=dict(size=13, color=SLATE),
            ),
            xaxis=dict(title="Price change scenario",
                       tickfont=dict(color=GREY_MD)),
            yaxis=dict(title="Δ Revenue vs baseline (£/week)",
                       tickfont=dict(color=GREY_MD)),
            height=340,
            margin=dict(t=50, b=40, l=60, r=40),
            plot_bgcolor="white",
            paper_bgcolor="white",
        )
        fig2.show()

        # ── Scenario table ────────────────────────────────────────────────
        display(HTML('<div style="font-size:.88rem;font-weight:600;'
                     f'color:{SLATE};margin:.8rem 0 .4rem;">What-if scenario table</div>'))

        def _fmt_delta(val, suffix=""):
            color = RED if val < 0 else GREEN
            sign  = "+" if val >= 0 else ""
            return (f'<span style="color:{color};font-weight:600;">'
                    f'{sign}{val}{suffix}</span>')

        rows_html = ""
        for _, row in sc.iterrows():
            bg = "#fef9f9" if row["pct"] > 0 else "#f9fef9"
            rows_html += (
                f'<tr style="background:{bg}">'
                f'<td style="font-weight:600">{row["Scenario"]}</td>'
                f'<td>£{row["New Price (£)"]:.2f}</td>'
                f'<td>{_fmt_delta(row["Δ Demand (units)"])}</td>'
                f'<td>{_fmt_delta(row["Δ Demand (%)"], "%")}</td>'
                f'<td>£{row["New Revenue (£)"]:,.0f}</td>'
                f'<td>{_fmt_delta(row["Δ Revenue (£)"], "")}</td>'
                f'<td>{_fmt_delta(row["Δ Revenue (%)"], "%")}</td>'
                f'</tr>'
            )

        th = ("background:#0f172a;color:white;padding:.6rem .9rem;"
              "font-size:.75rem;text-transform:uppercase;"
              "letter-spacing:.04em;font-weight:500;text-align:center;")
        td_style = ("font-size:.84rem;padding:.55rem .9rem;"
                    "text-align:center;border-bottom:1px solid #f1f5f9;")

        display(HTML(
            f'<div style="overflow-x:auto;margin-top:.5rem">'
            f'<table style="width:100%;border-collapse:collapse;">'
            f'<thead><tr>'
            f'<th style="{th}">Scenario</th>'
            f'<th style="{th}">New Price</th>'
            f'<th style="{th}">Δ Demand (units)</th>'
            f'<th style="{th}">Δ Demand (%)</th>'
            f'<th style="{th}">New Revenue</th>'
            f'<th style="{th}">Δ Revenue (£)</th>'
            f'<th style="{th}">Δ Revenue (%)</th>'
            f'</tr></thead>'
            f'<tbody style="{td_style}">{rows_html}</tbody>'
            f'</table></div>'
        ))

        # ── Model stats footnote ──────────────────────────────────────────
        display(HTML(
            f'<div style="font-size:.76rem;color:{GREY_MD};margin-top:.8rem;">'
            f'Scan*Pro model — β₁ (price elasticity) = {ε:.4f}  ·  '
            f'p-value = {r["elast_pval"]:.4f}  ·  '
            f'R² = {r["r2"]:.4f}  ·  '
            f'{"★ Statistically significant at p<0.05" if r["elast_sig"] else "Not significant at p<0.05"}'
            f'</div>'
        ))

sku_dd.observe(_render, names="value")
display(widgets.HBox([
    widgets.Label("Select SKU:", layout=widgets.Layout(
        margin="6px 8px 0 0", font_weight="bold")),
    sku_dd,
]))
display(out)
_render()


Output()

## Key findings & managerial insights

In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# ANSWER TO QUESTION: Which SKUs are most sensitive to price changes?
# ══════════════════════════════════════════════════════════════════════════════

display(HTML('<h3>Question 1: Which SKUs are most sensitive to price changes?</h3>'))

display(metric_row([
    metric_card('Overall range',
                f'{results_df["elasticity"].min():.2f} to {results_df["elasticity"].max():.2f}',
                f'Mean: {results_df["elasticity"].mean():.3f}'),
    metric_card('Highly Elastic (|ε|>2)',
                f'{(results_df["elast_category"]=="Highly Elastic").sum()}',
                'Very price-sensitive — promote, don\'t raise price'),
    metric_card('Moderately Elastic',
                f'{(results_df["elast_category"]=="Moderately Elastic").sum()}',
                '1 < |ε| < 2 — price changes matter'),
    metric_card('Inelastic (|ε|<1)',
                f'{(results_df["elast_category"]=="Inelastic").sum()}',
                'Price-insensitive — can raise price'),
]))

display(HTML('<h4>Top 5 most price-sensitive SKUs</h4>'))
top5 = results_df.nsmallest(5, 'elasticity')[['sku', 'elasticity', 'elast_pval', 'r2']].copy()
top5.columns = ['SKU', 'Elasticity', 'p-value', 'R²']
display(top5.style.format({'Elasticity': '{:.3f}', 'p-value': '{:.4f}', 'R²': '{:.3f}'}))

display(HTML('<h4>Top 5 least price-sensitive SKUs</h4>'))
bot5 = results_df.nlargest(5, 'elasticity')[['sku', 'elasticity', 'elast_pval', 'r2']].copy()
bot5.columns = ['SKU', 'Elasticity', 'p-value', 'R²']
display(bot5.style.format({'Elasticity': '{:.3f}', 'p-value': '{:.4f}', 'R²': '{:.3f}'}))

,SKU,Elasticity,p-value,R²
0,9,-4.071,0.0000,0.357
1,19,-4.007,0.0000,0.676
2,33,-3.511,0.0000,0.729
3,30,-2.927,0.0000,0.748
4,11,-2.924,0.0000,0.450


,SKU,Elasticity,p-value,R²
43,41,-0.374,0.2052,0.501
42,23,-0.443,0.3572,0.666
41,40,-0.478,0.0021,0.619
40,22,-0.598,0.0010,0.343
39,4,-0.747,0.0029,0.462


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# ANSWER TO QUESTION: How would a 10% price change impact demand and revenue?
# ══════════════════════════════════════════════════════════════════════════════

display(HTML('<h3>Question 2: Impact of a 10% price change across all SKUs</h3>'))

# Aggregate portfolio impact
total_base_rev = 0
total_rev_up = 0
total_rev_down = 0

impact_rows = []
for sku_id, r in results.items():
    P0, Q0 = r['avg_price'], r['avg_sales']
    e = r['elasticity']
    base_rev = P0 * Q0
    total_base_rev += base_rev
    
    P_up, P_dn = P0 * 1.10, P0 * 0.90
    Q_up = max(Q0 * ((P_up / P0) ** e), 0)
    Q_dn = max(Q0 * ((P_dn / P0) ** e), 0)
    R_up, R_dn = P_up * Q_up, P_dn * Q_dn
    total_rev_up += R_up
    total_rev_down += R_dn
    
    impact_rows.append({
        'SKU': sku_id,
        'Elasticity': round(e, 3),
        'Demand Δ (+10%)': round((Q_up - Q0) / Q0 * 100, 1),
        'Revenue Δ (+10%)': round((R_up - base_rev) / base_rev * 100, 1),
        'Revenue Δ (-10%)': round((R_dn - base_rev) / base_rev * 100, 1),
    })

pct_up = (total_rev_up - total_base_rev) / total_base_rev * 100
pct_dn = (total_rev_down - total_base_rev) / total_base_rev * 100

impact_df = pd.DataFrame(impact_rows)
n_gainers = (impact_df['Revenue Δ (+10%)'] > 0).sum()
n_losers = (impact_df['Revenue Δ (+10%)'] <= 0).sum()

display(metric_row([
    metric_card('Total weekly base revenue', f'£{total_base_rev:,.0f}', 'across all SKUs'),
    metric_card('+10% price → portfolio revenue', f'{pct_up:+.1f}%',
                f'£{total_rev_up:,.0f}/wk'),
    metric_card('-10% price → portfolio revenue', f'{pct_dn:+.1f}%',
                f'£{total_rev_down:,.0f}/wk'),
    metric_card('SKUs gaining from +10%', f'{n_gainers} / {len(impact_df)}',
                'inelastic — can absorb price rise'),
]))

display(HTML('<h4>Per-SKU impact of +10% price increase</h4>'))
display(impact_df.sort_values('Revenue Δ (+10%)', ascending=False).style.format({
    'Elasticity': '{:.3f}', 'Demand Δ (+10%)': '{:+.1f}%',
    'Revenue Δ (+10%)': '{:+.1f}%', 'Revenue Δ (-10%)': '{:+.1f}%'
}).background_gradient(subset=['Revenue Δ (+10%)'], cmap='RdYlGn'))

,SKU,Elasticity,Demand Δ (+10%),Revenue Δ (+10%),Revenue Δ (-10%)
40,41,-0.374,-3.5%,+6.1%,-6.4%
22,23,-0.443,-4.1%,+5.5%,-5.7%
39,40,-0.478,-4.5%,+5.1%,-5.4%
21,22,-0.598,-5.5%,+3.9%,-4.1%
3,4,-0.747,-6.9%,+2.4%,-2.6%
19,20,-0.756,-7.0%,+2.3%,-2.5%
41,42,-0.774,-7.1%,+2.2%,-2.4%
5,6,-0.798,-7.3%,+1.9%,-2.1%
20,21,-0.982,-8.9%,+0.2%,-0.2%
1,2,-0.977,-8.9%,+0.2%,-0.2%


### Managerial recommendations

1. **Price sensitivity varies widely across SKUs.** A one-size-fits-all pricing strategy would be suboptimal. Each SKU needs an individual pricing assessment based on its elasticity.

2. **Inelastic SKUs (|ε| < 1) are price-increase candidates.** For these SKUs, a 10% price rise causes less than 10% demand drop, so revenue increases. These are opportunities for margin improvement.

3. **Highly elastic SKUs (|ε| > 2) are promotion candidates.** Small price decreases generate disproportionately large demand increases. These SKUs benefit from promotional pricing to drive volume.

4. **Feature promotions provide additional lift.** The `feat_main_page` coefficient (β₂) is positive and significant for most SKUs, meaning featuring on the main page boosts sales independently of price. For elastic SKUs, combining a small discount with a feature promotion may be more effective than a large discount alone.

5. **Revenue-maximising ≠ profit-maximising.** The optimal prices shown are revenue-maximising. Without cost data, we cannot determine the profit-maximising price. Managers should use these results as a starting point and adjust for margins.

## Methodology

### Model choice

The task is to estimate **price elasticity** — how sensitive each SKU's demand is to price changes — and use it to simulate what-if pricing scenarios. Like the Promotion tab, this is a **coefficient estimation** problem: we need an accurate β₁, not a sales prediction.

We use Scan\*Pro log-log OLS because β₁ directly equals the price elasticity. OLS gives unbiased, interpretable estimates with standard errors for hypothesis testing.

### Specification

```
log(sales_t) = β₀ + β₁·log(price_t) + β₂·feat_main_page_t + β₃·trend_t + Σ γ_m·month_m + ε_t
```

| Feature | Why included |
|---------|-------------|
| log(price) | Target variable — β₁ = elasticity directly |
| feat_main_page | Controls for simultaneous promotions |
| trend | Controls for secular demand drift |
| month dummies | Controls for seasonal demand patterns |

### Why no lagged prices?

The Promotion tab includes lagged prices to avoid bias on β₂. This tab excludes them because:
1. The goal is **steady-state** elasticity for sustained price changes, not short-run vs long-run decomposition
2. Scenario simulation needs a single elasticity — including lags fragments it into components that are ambiguous to propagate forward
3. The contemporaneous specification follows the standard approach for price sensitivity estimation in the Scan\*Pro framework

### Scenario simulation

| Metric | Formula |
|--------|--------|
| New demand | Q₀ × (P_new / P₀)^ε |
| New revenue | P_new × Q_new |
| Elastic (\|ε\| > 1) | Price cut → revenue grows |
| Inelastic (\|ε\| < 1) | Price rise → revenue grows |

### Limitations

- Constant elasticity assumption — may not hold at extreme prices
- Revenue-maximising, not profit-maximising (no cost/margin data)
- Cross-price effects not modelled
- SKUs with limited price variation may have unreliable estimates
- Endogeneity: retailers may set prices based on expected demand (simultaneity bias)

### References

- Van Heerde, H.J., Leeflang, P.S.H., & Wittink, D.R. (2002). *Schmalenbach Business Review*, 54, 198–220.
- Hanssens, D.M., Parsons, L.J., & Schultz, R.L. (2001). *Market Response Models*, 2nd edition. Kluwer.